In [1]:
# pip install xgboost scikit-learn

import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def build_xgboost_binary_classifier(
    n_estimators: int = 800,
    learning_rate: float = 0.05,
    max_depth: int = 5,
    subsample: float = 0.8,
    colsample_bytree: float = 0.8,
    reg_lambda: float = 1.0,
    reg_alpha: float = 0.0,
    min_child_weight: float = 1.0,
    gamma: float = 0.0,
    random_state: int = 42,
    n_jobs: int = -1,
):
    """
    Replacement for your PyTorch BinaryClassifier:
    Returns an XGBoost binary classifier (sklearn API).
    """
    model = XGBClassifier(
        objective="binary:logistic",   # outputs probabilities via sigmoid
        eval_metric="logloss",         # avoid warning; can also use "auc"
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_lambda=reg_lambda,
        reg_alpha=reg_alpha,
        min_child_weight=min_child_weight,
        gamma=gamma,
        random_state=random_state,
        n_jobs=n_jobs,
        tree_method="hist",            # fast + good default on CPU
    )
    return model

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# 0) Load ALREADY-SCALED binary dataset
df_binary = pd.read_csv("artifacts/final_train_scaled_binary_clustered.csv")

TARGET_COL = "MP_binary_high"

exclude = {"SMILES", TARGET_COL, "MW", "MP", "MP_category_default", "Structure_Cluster"}

num_cols = df_binary.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

# ---- NEW: load selected feature names (produced by your RFE step) ----
selected_feature_names = pd.read_csv("selected_rdkit_features.csv", header=None)[0].tolist()

# Keep only selected features that exist in this dataframe (safety)
selected_feature_names = [c for c in selected_feature_names if c in feature_cols]

# Build reduced X/y
X = df_binary[selected_feature_names].to_numpy(np.float32)
y = df_binary[TARGET_COL].to_numpy(np.int32) 

# Stratification labels (cluster)
y_strat = df_binary["Structure_Cluster"].astype(str).to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("n features:", len(selected_feature_names))
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))

# 1) Fix folds once (same)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = [(tr_idx, val_idx) for tr_idx, val_idx in skf.split(X, y_strat)]
print("Built folds:", len(folds))

X shape: (17625, 20)
y shape: (17625,)
n features: 20
Label counts: {np.int32(0): np.int64(16853), np.int32(1): np.int64(772)}
Built folds: 10


In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
import joblib


def evaluate_fold_xgb(
    fold_idx,
    X_train,
    y_train,
    X_val,
    y_val,
    params,
    save_checkpoints=True,
    checkpoint_dir="checkpoints_xgb_classifier",
):
    y_train = np.asarray(y_train).astype(int)
    y_val   = np.asarray(y_val).astype(int)

    # imbalance knob: neg/pos (train fold only)
    pos = y_train.sum()
    neg = y_train.size - pos
    scale_pos_weight = float(neg / (pos + 1e-12))

    model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    early_stopping_rounds=50,   # <-- HERE
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    **params,
)

    model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)


    probs = model.predict_proba(X_val)[:, 1]
    preds = (probs >= 0.5).astype(int)

    auc = roc_auc_score(y_val, probs)
    val_loss = log_loss(y_val, probs, labels=[0, 1])
    acc = accuracy_score(y_val, preds)

    best_iter = getattr(model, "best_iteration", None)

    if save_checkpoints:
        ckpt_root = Path(checkpoint_dir) / f"fold_{fold_idx}"
        ckpt_root.mkdir(parents=True, exist_ok=True)

        # Avoid overwriting: include fold in filename
        joblib.dump(model, ckpt_root / f"xgb_model_fold_{fold_idx}.joblib")

        pd.DataFrame([{
            "fold": fold_idx,
            "val_auc": float(auc),
            "val_logloss": float(val_loss),
            "accuracy@0.5": float(acc),
            "best_iteration": best_iter,
            "scale_pos_weight": scale_pos_weight,
        }]).to_csv(ckpt_root / f"metrics_fold_{fold_idx}.csv", index=False)

    print(
        f"[Fold {fold_idx}] AUC: {auc:.4f} | Logloss: {val_loss:.4f} | Acc@0.5: {acc:.4f} | best_iter: {best_iter}"
    )
    return float(auc), float(val_loss), float(acc), model, best_iter


import optuna

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 50.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True),
    }

    aucs, losses, accs = [], [], []

    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        X_train = X[tr_idx]
        y_train = y[tr_idx]
        X_val   = X[val_idx]
        y_val   = y[val_idx]

        trial_checkpoint_root = Path("checkpoints_xgb_classifier") / f"trial_{trial.number:03d}"

        auc, val_loss, acc, _, _ = evaluate_fold_xgb(
            fold_idx=fold_idx,
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            params=params,
            save_checkpoints=True,
            checkpoint_dir=trial_checkpoint_root,
        )

        aucs.append(auc)
        losses.append(val_loss)
        accs.append(acc)

    avg_auc = float(np.mean(aucs))
    avg_loss = float(np.mean(losses))
    avg_acc = float(np.mean(accs))

    print(
        f"Trial {trial.number}: Avg AUC={avg_auc:.4f} | Avg Logloss={avg_loss:.4f} | Avg Acc@0.5={avg_acc:.4f}"
    )

    # Optuna maximizes AUC (recommended)
    return avg_auc


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best params:", study.best_params)
print("Best value (AUC):", study.best_value)


[I 2026-01-19 03:01:10,756] A new study created in memory with name: no-name-73c835d0-6174-4b23-818f-43dcfdf02be3


[Fold 0] AUC: 0.9439 | Logloss: 0.2478 | Acc@0.5: 0.8860 | best_iter: 917
[Fold 1] AUC: 0.9425 | Logloss: 0.2740 | Acc@0.5: 0.8729 | best_iter: 657
[Fold 2] AUC: 0.9286 | Logloss: 0.3505 | Acc@0.5: 0.8395 | best_iter: 167
[Fold 3] AUC: 0.9172 | Logloss: 0.3320 | Acc@0.5: 0.8565 | best_iter: 195
[Fold 4] AUC: 0.9197 | Logloss: 0.2944 | Acc@0.5: 0.8707 | best_iter: 362
[Fold 5] AUC: 0.9370 | Logloss: 0.2526 | Acc@0.5: 0.8848 | best_iter: 1014
[Fold 6] AUC: 0.9365 | Logloss: 0.2490 | Acc@0.5: 0.8944 | best_iter: 1140
[Fold 7] AUC: 0.9283 | Logloss: 0.3096 | Acc@0.5: 0.8536 | best_iter: 380
[Fold 8] AUC: 0.9310 | Logloss: 0.3315 | Acc@0.5: 0.8479 | best_iter: 226


[I 2026-01-19 03:01:18,004] Trial 0 finished with value: 0.9304262703871873 and parameters: {'n_estimators': 2324, 'learning_rate': 0.020894160431898266, 'max_depth': 4, 'min_child_weight': 15.125605441002925, 'subsample': 0.9767017405354007, 'colsample_bytree': 0.5513760480397957, 'gamma': 3.8487550297691757, 'reg_lambda': 0.005270803920141563, 'reg_alpha': 9.610510663679555e-05}. Best is trial 0 with value: 0.9304262703871873.


[Fold 9] AUC: 0.9195 | Logloss: 0.3207 | Acc@0.5: 0.8575 | best_iter: 248
Trial 0: Avg AUC=0.9304 | Avg Logloss=0.2962 | Avg Acc@0.5=0.8664
[Fold 0] AUC: 0.9529 | Logloss: 0.1704 | Acc@0.5: 0.9280 | best_iter: 366
[Fold 1] AUC: 0.9462 | Logloss: 0.2249 | Acc@0.5: 0.9019 | best_iter: 142
[Fold 2] AUC: 0.9374 | Logloss: 0.2708 | Acc@0.5: 0.8809 | best_iter: 64
[Fold 3] AUC: 0.9247 | Logloss: 0.2519 | Acc@0.5: 0.8928 | best_iter: 85
[Fold 4] AUC: 0.9250 | Logloss: 0.2304 | Acc@0.5: 0.9075 | best_iter: 114
[Fold 5] AUC: 0.9353 | Logloss: 0.2572 | Acc@0.5: 0.8814 | best_iter: 93
[Fold 6] AUC: 0.9401 | Logloss: 0.2138 | Acc@0.5: 0.9154 | best_iter: 217
[Fold 7] AUC: 0.9316 | Logloss: 0.2661 | Acc@0.5: 0.8774 | best_iter: 76
[Fold 8] AUC: 0.9378 | Logloss: 0.2552 | Acc@0.5: 0.8871 | best_iter: 88


[I 2026-01-19 03:01:21,255] Trial 1 finished with value: 0.9350527433024215 and parameters: {'n_estimators': 878, 'learning_rate': 0.07040711531516214, 'max_depth': 6, 'min_child_weight': 18.471811247133786, 'subsample': 0.5474768103642326, 'colsample_bytree': 0.589522390675819, 'gamma': 4.221704429442791, 'reg_lambda': 0.002270356976819487, 'reg_alpha': 1.7713127648232698}. Best is trial 1 with value: 0.9350527433024215.


...

[Fold 9] AUC: 0.9219 | Logloss: 0.1699 | Acc@0.5: 0.9262 | best_iter: 139
Trial 17: Avg AUC=0.9345 | Avg Logloss=0.1921 | Avg Acc@0.5=0.9217
[Fold 0] AUC: 0.9394 | Logloss: 0.2498 | Acc@0.5: 0.8843 | best_iter: 45
[Fold 1] AUC: 0.9441 | Logloss: 0.1675 | Acc@0.5: 0.9319 | best_iter: 166
[Fold 2] AUC: 0.9303 | Logloss: 0.2296 | Acc@0.5: 0.9030 | best_iter: 68
[Fold 3] AUC: 0.9219 | Logloss: 0.2484 | Acc@0.5: 0.8883 | best_iter: 43
[Fold 4] AUC: 0.9160 | Logloss: 0.2189 | Acc@0.5: 0.9109 | best_iter: 67
[Fold 5] AUC: 0.9420 | Logloss: 0.1856 | Acc@0.5: 0.9183 | best_iter: 132
[Fold 6] AUC: 0.9359 | Logloss: 0.2404 | Acc@0.5: 0.8939 | best_iter: 61
[Fold 7] AUC: 0.9331 | Logloss: 0.2352 | Acc@0.5: 0.8927 | best_iter: 67
[Fold 8] AUC: 0.9316 | Logloss: 0.2671 | Acc@0.5: 0.8882 | best_iter: 39


[I 2026-01-19 03:02:45,496] Trial 18 finished with value: 0.931397596050793 and parameters: {'n_estimators': 1106, 'learning_rate': 0.08976435765872764, 'max_depth': 6, 'min_child_weight': 5.814452865402799, 'subsample': 0.7752892097439209, 'colsample_bytree': 0.9353966462634985, 'gamma': 1.8085447631703297, 'reg_lambda': 0.16802253139463175, 'reg_alpha': 0.0003660237948995472}. Best is trial 10 with value: 0.9366294281009996.


[Fold 9] AUC: 0.9196 | Logloss: 0.1939 | Acc@0.5: 0.9154 | best_iter: 115
Trial 18: Avg AUC=0.9314 | Avg Logloss=0.2236 | Avg Acc@0.5=0.9027
[Fold 0] AUC: 0.9530 | Logloss: 0.1321 | Acc@0.5: 0.9467 | best_iter: 100
[Fold 1] AUC: 0.9426 | Logloss: 0.1433 | Acc@0.5: 0.9444 | best_iter: 85
[Fold 2] AUC: 0.9346 | Logloss: 0.1549 | Acc@0.5: 0.9325 | best_iter: 72
[Fold 3] AUC: 0.9243 | Logloss: 0.2093 | Acc@0.5: 0.9217 | best_iter: 20
[Fold 4] AUC: 0.9186 | Logloss: 0.1477 | Acc@0.5: 0.9518 | best_iter: 140
[Fold 5] AUC: 0.9387 | Logloss: 0.1465 | Acc@0.5: 0.9404 | best_iter: 101
[Fold 6] AUC: 0.9393 | Logloss: 0.1699 | Acc@0.5: 0.9330 | best_iter: 59
[Fold 7] AUC: 0.9362 | Logloss: 0.1967 | Acc@0.5: 0.9200 | best_iter: 38
[Fold 8] AUC: 0.9297 | Logloss: 0.2103 | Acc@0.5: 0.9058 | best_iter: 27


[I 2026-01-19 03:02:48,889] Trial 19 finished with value: 0.9338267246488302 and parameters: {'n_estimators': 762, 'learning_rate': 0.17852422458593126, 'max_depth': 9, 'min_child_weight': 13.308373242791912, 'subsample': 0.8315002433944294, 'colsample_bytree': 0.8584015616521465, 'gamma': 0.12218165996531316, 'reg_lambda': 3.30995288308445, 'reg_alpha': 8.325187703539956e-06}. Best is trial 10 with value: 0.9366294281009996.


[Fold 9] AUC: 0.9213 | Logloss: 0.2076 | Acc@0.5: 0.9103 | best_iter: 20
Trial 19: Avg AUC=0.9338 | Avg Logloss=0.1718 | Avg Acc@0.5=0.9307
Best params: {'n_estimators': 1162, 'learning_rate': 0.023962556929236718, 'max_depth': 10, 'min_child_weight': 8.99775502964076, 'subsample': 0.8106626806922802, 'colsample_bytree': 0.9881283795532174, 'gamma': 1.847361653297833, 'reg_lambda': 0.9569699786903291, 'reg_alpha': 2.5490266596638967e-06}
Best value (AUC): 0.9366294281009996


In [7]:
best_params = study.best_params
print(best_params)

{'n_estimators': 1162, 'learning_rate': 0.023962556929236718, 'max_depth': 10, 'min_child_weight': 8.99775502964076, 'subsample': 0.8106626806922802, 'colsample_bytree': 0.9881283795532174, 'gamma': 1.847361653297833, 'reg_lambda': 0.9569699786903291, 'reg_alpha': 2.5490266596638967e-06}


In [8]:
import numpy as np
import pandas as pd
from pathlib import Path

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, confusion_matrix
import joblib

BASE = Path.cwd()
artifacts_dir = BASE / "artifacts"

best_models_dir = artifacts_dir / "xgb_best_models"
best_models_dir.mkdir(parents=True, exist_ok=True)

final_ckpt_dir = BASE / "checkpoints_xgb_best"
final_ckpt_dir.mkdir(parents=True, exist_ok=True)

print("Best hyperparameters from Optuna:", best_params)

final_fold_metrics = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n==================== Final XGBoost training for fold {fold_idx} ====================")

    X_train = X[tr_idx]
    y_train = np.asarray(y[tr_idx]).astype(int)

    X_val = X[val_idx]
    y_val = np.asarray(y[val_idx]).astype(int)

    # scale_pos_weight (train fold only)
    pos = y_train.sum()
    neg = y_train.size - pos
    scale_pos_weight = float(neg / (pos + 1e-12))

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",            # aligns with optimizing AUC
        early_stopping_rounds=50,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        **best_params,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    probs = model.predict_proba(X_val)[:, 1]
    preds = (probs >= 0.5).astype(int)

    auc = roc_auc_score(y_val, probs)
    ll  = log_loss(y_val, probs, labels=[0, 1])
    acc = accuracy_score(y_val, preds)

    cm = confusion_matrix(y_val, preds)
    tn, fp, fn, tp = cm.ravel()

    best_iter = getattr(model, "best_iteration", None)

    # Save model (per fold)
    model_path = best_models_dir / f"xgb_best_fold_{fold_idx}.joblib"
    joblib.dump(model, model_path)

    print(f"[Fold {fold_idx}] Saved model to: {model_path}")
    print(f"[Fold {fold_idx}] AUC: {auc:.4f} | Logloss: {ll:.4f} | Acc@0.5: {acc:.4f} | best_iter: {best_iter}")
    print(f"[Fold {fold_idx}] CM TN={tn} FP={fp} FN={fn} TP={tp}")

    final_fold_metrics.append({
        "Fold": fold_idx,
        "AUC": float(auc),
        "Logloss": float(ll),
        "Accuracy@0.5": float(acc),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Best_Iteration": best_iter,
        "Scale_Pos_Weight": scale_pos_weight,
        "Model_Path": str(model_path),
    })

metrics_df = pd.DataFrame(final_fold_metrics)
metrics_path = best_models_dir / "xgb_best_models_summary.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\n✅ Saved summary of best XGBoost models across folds to:", metrics_path)
print(metrics_df)


Best hyperparameters from Optuna: {'n_estimators': 1162, 'learning_rate': 0.023962556929236718, 'max_depth': 10, 'min_child_weight': 8.99775502964076, 'subsample': 0.8106626806922802, 'colsample_bytree': 0.9881283795532174, 'gamma': 1.847361653297833, 'reg_lambda': 0.9569699786903291, 'reg_alpha': 2.5490266596638967e-06}

==================== Final XGBoost training for fold 0 ====================
[Fold 0] Saved model to: /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/artifacts/xgb_best_models/xgb_best_fold_0.joblib
[Fold 0] AUC: 0.9507 | Logloss: 0.1311 | Acc@0.5: 0.9484 | best_iter: 502
[Fold 0] CM TN=1620 FP=66 FN=25 TP=52

==================== Final XGBoost training for fold 1 ====================
[Fold 1] Saved model to: /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/artifacts/xgb_best_models/xgb_best_fold_1.joblib
[Fold 1] AUC: 0.9456 | Logloss: 0.1445 | Acc@0.5: 0.9433 | best_iter: 315
[Fold 1] CM TN=1617 FP=76 FN=24 TP=46

==================== Final XGBoost training for 

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix
import joblib

confusion_matrices = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n===== Confusion matrix for fold {fold_idx} =====")

    # Validation data
    X_val = X[val_idx]
    y_val = y[val_idx].astype(int)

    # Load best XGBoost model for this fold
    model_path = best_models_dir / f"xgb_best_fold_{fold_idx}.joblib"
    model = joblib.load(model_path)

    # Predict probabilities (class 1 = high MP)
    probs = model.predict_proba(X_val)[:, 1]
    preds = (probs >= 0.5).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y_val, preds, labels=[0, 1])
    confusion_matrices.append(cm)

    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")



===== Confusion matrix for fold 0 =====
Confusion matrix (rows=true, cols=pred):
[[1620   66]
 [  25   52]]
TN=1620, FP=66, FN=25, TP=52

===== Confusion matrix for fold 1 =====
Confusion matrix (rows=true, cols=pred):
[[1617   76]
 [  24   46]]
TN=1617, FP=76, FN=24, TP=46

===== Confusion matrix for fold 2 =====
Confusion matrix (rows=true, cols=pred):
[[1596   89]
 [  24   54]]
TN=1596, FP=89, FN=24, TP=54

===== Confusion matrix for fold 3 =====
Confusion matrix (rows=true, cols=pred):
[[1615   86]
 [  17   45]]
TN=1615, FP=86, FN=17, TP=45

===== Confusion matrix for fold 4 =====
Confusion matrix (rows=true, cols=pred):
[[1614   60]
 [  37   52]]
TN=1614, FP=60, FN=37, TP=52

===== Confusion matrix for fold 5 =====
Confusion matrix (rows=true, cols=pred):
[[1612   75]
 [  27   48]]
TN=1612, FP=75, FN=27, TP=48

===== Confusion matrix for fold 6 =====
Confusion matrix (rows=true, cols=pred):
[[1573   88]
 [  26   75]]
TN=1573, FP=88, FN=26, TP=75

===== Confusion matrix for fold 7

In [10]:
import numpy as np
import pandas as pd

mean_cm = np.mean(confusion_matrices, axis=0)

mean_cm_df = pd.DataFrame(
    mean_cm,
    index=["True Low&Int", "True High"],
    columns=["Pred Low&Int", "Pred High"],
)

print("\n===== Mean confusion matrix across folds (XGBoost) =====")
print(mean_cm_df)

# Save to CSV (use an XGB-specific name)
mean_cm_path = best_models_dir / "xgb_mean_confusion_matrix.csv"
mean_cm_df.to_csv(mean_cm_path, index=True)
print(f"\n✅ Saved mean confusion matrix to: {mean_cm_path}")



===== Mean confusion matrix across folds (XGBoost) =====
              Pred Low&Int  Pred High
True Low&Int        1600.4       84.9
True High             24.8       52.4

✅ Saved mean confusion matrix to: /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/artifacts/xgb_best_models/xgb_mean_confusion_matrix.csv


In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import joblib

def metrics_from_confusion(cm):
    """
    cm = [[TN, FP],
          [FN, TP]]
    """
    TN, FP, FN, TP = cm.ravel()

    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Specificity": specificity,
    }

all_fold_metrics = []
all_conf_matrices = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\nEvaluating fold {fold_idx}")

    # Load best XGBoost model for this fold
    model_path = best_models_dir / f"xgb_best_fold_{fold_idx}.joblib"
    model = joblib.load(model_path)

    # Validation data
    X_val = X[val_idx]
    y_val = y[val_idx].astype(int)

    # Predict probabilities + labels
    probs = model.predict_proba(X_val)[:, 1]
    preds = (probs >= 0.5).astype(int)

    # Confusion matrix: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_val, preds, labels=[0, 1])
    all_conf_matrices.append(cm)

    # Metrics
    metrics = metrics_from_confusion(cm)
    metrics["Fold"] = fold_idx
    all_fold_metrics.append(metrics)

    print("Confusion matrix:\n", cm)
    print(metrics)

# Optional: turn metrics into a DataFrame
df_metrics = pd.DataFrame(all_fold_metrics)
print("\nPer-fold metrics:")
print(df_metrics)

# Optional: mean confusion matrix across folds
mean_cm = np.mean(all_conf_matrices, axis=0)
mean_cm_df = pd.DataFrame(mean_cm, index=["True 0", "True 1"], columns=["Pred 0", "Pred 1"])
print("\nMean confusion matrix across folds:")
print(mean_cm_df)



Evaluating fold 0
Confusion matrix:
 [[1620   66]
 [  25   52]]
{'Accuracy': np.float64(0.9483834373227453), 'Precision': np.float64(0.4406779661016949), 'Recall': np.float64(0.6753246753246753), 'F1': np.float64(0.5333333333333333), 'Specificity': np.float64(0.9608540925266904), 'Fold': 0}

Evaluating fold 1
Confusion matrix:
 [[1617   76]
 [  24   46]]
{'Accuracy': np.float64(0.9432785025524674), 'Precision': np.float64(0.3770491803278688), 'Recall': np.float64(0.6571428571428571), 'F1': np.float64(0.47916666666666663), 'Specificity': np.float64(0.9551092734790313), 'Fold': 1}

Evaluating fold 2
Confusion matrix:
 [[1596   89]
 [  24   54]]
{'Accuracy': np.float64(0.9359047078842881), 'Precision': np.float64(0.3776223776223776), 'Recall': np.float64(0.6923076923076923), 'F1': np.float64(0.48868778280542974), 'Specificity': np.float64(0.9471810089020771), 'Fold': 2}

Evaluating fold 3
Confusion matrix:
 [[1615   86]
 [  17   45]]
{'Accuracy': np.float64(0.9415768576290414), 'Precisio

In [12]:
metrics_df = pd.DataFrame(all_fold_metrics)

mean_metrics = metrics_df.mean(numeric_only=True)
std_metrics  = metrics_df.std(numeric_only=True)

summary_df = pd.DataFrame({
    "Mean": mean_metrics,
    "Std": std_metrics
})

print("\n=== Cross-validated performance ===")
print(summary_df)



=== Cross-validated performance ===
                 Mean       Std
Accuracy     0.937757  0.010041
Precision    0.385367  0.054430
Recall       0.679058  0.045356
F1           0.488550  0.041949
Specificity  0.949620  0.011303
Fold         4.500000  3.027650
